# 20 — Feature Engineering

**ENAI 603 Capstone — WMATA Metro Delay Prediction**

This notebook performs further feature engineering on the WMATA database with the aim of improving models performance.

**Sections:**
1. Load existing features.csv and WMATA Stations dataset files
2. Add station has parking feature
3. Add transfer station feature
4. Add VRE, AMTRAK, MARK connection feature
5. Add number of rail systems connection (interaction term) feature
6. Add previous and next stops (locations) features
7. Save new features into existing features.csv

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd
import numpy as np

from fuzzywuzzy import process
from functools import lru_cache

PROJECT_DIR = Path("..")
DB_PATH = PROJECT_DIR / "data" / "wmata.db"

conn = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH}")
print(f"Database size: {DB_PATH.stat().st_size / 1024:.0f} KB")

Connected to ../data/wmata.db
Database size: 868640 KB


## 1. Load existing features.csv and WMATA Stations dataset files

In [2]:
# Load initial features.csv file
df_features = pd.read_csv(PROJECT_DIR / "data" / "features.csv")
df_features.head()

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,lon,minutes_num,num_trains_at_station,avg_cars_at_station,delay_rate_30min,line_delay_rate_30min,active_incident,incident_is_delay,scheduled_headway_min,is_delayed
0,2026-03-11 02:00:05.155939+00:00,Metro Center,Glenmont,2026-03-10,22,1,0,0,RD,A01,...,-77.028099,1.0,6,6.666667,0.0,0.0,0,0,0.0,0
1,2026-03-11 02:00:05.155939+00:00,Court House,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,K01,...,-77.083910,4.0,6,7.333333,0.0,0.0,1,1,NaN,0
2,2026-03-11 02:00:05.155939+00:00,Benning Road,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G01,...,-76.938291,2.0,6,7.000000,0.0,0.0,0,0,NaN,0
3,2026-03-11 02:00:05.155939+00:00,Addison Road-Seat Pleasant,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G03,...,-76.893592,26.0,6,7.333333,0.0,0.0,0,0,NaN,0
4,2026-03-11 02:00:05.155939+00:00,Van Dorn Street,Franconia-Springfield,2026-03-10,22,1,0,0,BL,J02,...,-77.129407,30.0,6,6.000000,0.0,0.0,0,0,NaN,0


In [3]:
# Load stations dataset
df_stations = pd.read_sql("SELECT * FROM stations", conn)
print(f"Total stations: {len(df_stations)}")
df_stations

Total stations: 102


,station_code,station_name,lat,lon,line_code1,line_code2,line_code3,line_code4,together_station
0,A01,Metro Center,38.898303,-77.028099,RD,NaN,NaN,None,C01
1,A02,Farragut North,38.903192,-77.039766,RD,NaN,NaN,None,
2,A03,Dupont Circle,38.909499,-77.043620,RD,NaN,NaN,None,
3,A04,Woodley Park-Zoo/Adams Morgan,38.924999,-77.052648,RD,NaN,NaN,None,
4,A05,Cleveland Park,38.934703,-77.058226,RD,NaN,NaN,None,
...,...,...,...,...,...,...,...,...,...
97,N08,Herndon,38.952821,-77.385178,SV,NaN,NaN,None,
98,N09,Innovation Center,38.960758,-77.415295,SV,NaN,NaN,None,
99,N10,Washington Dulles International Airport,38.955784,-77.448148,SV,NaN,NaN,None,
100,N11,Loudoun Gateway,38.992040,-77.460685,SV,NaN,NaN,None,


---

# Feature Engineering

## 2. Add station has parking feature

In [4]:
STATION_PARKING = ['N12', 'N11', 'N09', 'N08', 'N06', 'K05', 'K08', 'K07', 'K06', 'A15', 'A14', 'A13', 'A12', 'A11', 'J03', 'J02', 'C15',
                   'F11', 'F10', 'F09', 'F08', 'F06', 'G02', 'G03', 'G04', 'G05', 'D09', 'D10', 'D11', 'D12', 'D13',
                   'B04', 'B07', 'B06', 'B08', 'B09', 'B10', 'B11', 'E07', 'E08', 'E09', 'E10']

# Check if the upcoming/current station location has parking availability
def station_parking_available(location_code):
  if location_code in STATION_PARKING:
    return 1
  else:
    return 0

# Add loc_has_parking feature based on location_code
df_features['loc_has_parking'] = df_features['location_code'].apply(station_parking_available)
df_features

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,minutes_num,num_trains_at_station,avg_cars_at_station,delay_rate_30min,line_delay_rate_30min,active_incident,incident_is_delay,scheduled_headway_min,is_delayed,loc_has_parking
0,2026-03-11 02:00:05.155939+00:00,Metro Center,Glenmont,2026-03-10,22,1,0,0,RD,A01,...,1.0,6,6.666667,0.0,0.000000,0,0,0.0,0,0
1,2026-03-11 02:00:05.155939+00:00,Court House,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,K01,...,4.0,6,7.333333,0.0,0.000000,1,1,NaN,0,0
2,2026-03-11 02:00:05.155939+00:00,Benning Road,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G01,...,2.0,6,7.000000,0.0,0.000000,0,0,NaN,0,0
3,2026-03-11 02:00:05.155939+00:00,Addison Road-Seat Pleasant,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G03,...,26.0,6,7.333333,0.0,0.000000,0,0,NaN,0,1
4,2026-03-11 02:00:05.155939+00:00,Van Dorn Street,Franconia-Springfield,2026-03-10,22,1,0,0,BL,J02,...,30.0,6,6.000000,0.0,0.000000,0,0,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333137,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,3.0,5,7.500000,0.0,0.134638,1,1,0.5,0,0
333138,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,0.0,5,7.500000,0.0,0.134483,1,1,0.5,0,0
333139,2026-03-18 18:30:16.166019+00:00,Farragut North,Shady Grv,2026-03-18,14,2,0,0,RD,A02,...,2.0,6,7.333333,0.0,0.134328,0,0,NaN,0,0
333140,2026-03-18 18:30:16.166019+00:00,Takoma,Glenmont,2026-03-18,14,2,0,0,RD,B07,...,12.0,6,7.600000,0.0,0.134174,0,0,1.5,0,1


## 3. Add transfer station feature

In [5]:
TRANSFER_STATIONS = ['K05', 'C05', 'C07', 'C13', 'A01', 'C01', 'B01', 'F01', 'D03', 'F03', 'E01', 'B06', 'E06' 'D08']

# Check if the upcoming/current station location is a transfer station
def is_transfer_station(location_code):
  if location_code in TRANSFER_STATIONS:
    return 1
  else:
    return 0

# Add loc_is_transfer feature based on location_code
df_features['loc_is_transfer'] = df_features['location_code'].apply(is_transfer_station)
df_features

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,num_trains_at_station,avg_cars_at_station,delay_rate_30min,line_delay_rate_30min,active_incident,incident_is_delay,scheduled_headway_min,is_delayed,loc_has_parking,loc_is_transfer
0,2026-03-11 02:00:05.155939+00:00,Metro Center,Glenmont,2026-03-10,22,1,0,0,RD,A01,...,6,6.666667,0.0,0.000000,0,0,0.0,0,0,1
1,2026-03-11 02:00:05.155939+00:00,Court House,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,K01,...,6,7.333333,0.0,0.000000,1,1,NaN,0,0,0
2,2026-03-11 02:00:05.155939+00:00,Benning Road,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G01,...,6,7.000000,0.0,0.000000,0,0,NaN,0,0,0
3,2026-03-11 02:00:05.155939+00:00,Addison Road-Seat Pleasant,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G03,...,6,7.333333,0.0,0.000000,0,0,NaN,0,1,0
4,2026-03-11 02:00:05.155939+00:00,Van Dorn Street,Franconia-Springfield,2026-03-10,22,1,0,0,BL,J02,...,6,6.000000,0.0,0.000000,0,0,NaN,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333137,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,5,7.500000,0.0,0.134638,1,1,0.5,0,0,1
333138,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,5,7.500000,0.0,0.134483,1,1,0.5,0,0,1
333139,2026-03-18 18:30:16.166019+00:00,Farragut North,Shady Grv,2026-03-18,14,2,0,0,RD,A02,...,6,7.333333,0.0,0.134328,0,0,NaN,0,0,0
333140,2026-03-18 18:30:16.166019+00:00,Takoma,Glenmont,2026-03-18,14,2,0,0,RD,B07,...,6,7.600000,0.0,0.134174,0,0,1.5,0,1,0


## 4. Add VRE, AMTRAK, MARK connection feature

In [6]:
RAIL_SYS_CONNECT = {
  'VRE': ['B03', 'C13', 'C09', 'J03', 'D03', 'F03'],
  'AMTRAK': ['D13', 'B03', 'C13', 'A14'],
  'MARK':  ['D13', 'E10', 'E09', 'B03', 'B08', 'A14']
}

# Check if the upcoming/current station location connects to any other rail systems
def connect_rail_systems(location_code):
  connects = []
  
  for rail, stations in RAIL_SYS_CONNECT.items():
    if location_code in stations:
      connects.append(1)
    else:
      connects.append(0)
  
  return connects

# Add VRE, AMTRAK, MARK connection feature based on location_code
df_features[['loc_conn_vre', 'loc_conn_amtrak', 'loc_conn_mark']] = df_features['location_code'].apply(connect_rail_systems).to_list()
df_features

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,line_delay_rate_30min,active_incident,incident_is_delay,scheduled_headway_min,is_delayed,loc_has_parking,loc_is_transfer,loc_conn_vre,loc_conn_amtrak,loc_conn_mark
0,2026-03-11 02:00:05.155939+00:00,Metro Center,Glenmont,2026-03-10,22,1,0,0,RD,A01,...,0.000000,0,0,0.0,0,0,1,0,0,0
1,2026-03-11 02:00:05.155939+00:00,Court House,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,K01,...,0.000000,1,1,NaN,0,0,0,0,0,0
2,2026-03-11 02:00:05.155939+00:00,Benning Road,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G01,...,0.000000,0,0,NaN,0,0,0,0,0,0
3,2026-03-11 02:00:05.155939+00:00,Addison Road-Seat Pleasant,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G03,...,0.000000,0,0,NaN,0,1,0,0,0,0
4,2026-03-11 02:00:05.155939+00:00,Van Dorn Street,Franconia-Springfield,2026-03-10,22,1,0,0,BL,J02,...,0.000000,0,0,NaN,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333137,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,0.134638,1,1,0.5,0,0,1,0,0,0
333138,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,0.134483,1,1,0.5,0,0,1,0,0,0
333139,2026-03-18 18:30:16.166019+00:00,Farragut North,Shady Grv,2026-03-18,14,2,0,0,RD,A02,...,0.134328,0,0,NaN,0,0,0,0,0,0
333140,2026-03-18 18:30:16.166019+00:00,Takoma,Glenmont,2026-03-18,14,2,0,0,RD,B07,...,0.134174,0,0,1.5,0,1,0,0,0,0


## 5. Add Number of Rail systems connection feature

In [7]:
# Add interaction term "num_rails_conn" based on sum of VRE, AMTRAK, MARK connections in a location
df_features['num_rails_conn'] = df_features['loc_conn_vre'] + df_features['loc_conn_amtrak'] + df_features['loc_conn_mark']
df_features[df_features['num_rails_conn'] > 1]

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,active_incident,incident_is_delay,scheduled_headway_min,is_delayed,loc_has_parking,loc_is_transfer,loc_conn_vre,loc_conn_amtrak,loc_conn_mark,num_rails_conn
177,2026-03-11 02:00:05.155939+00:00,New Carrollton,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,D13,...,1,1,NaN,0,1,0,0,1,1,2
183,2026-03-11 02:00:05.155939+00:00,New Carrollton,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,D13,...,1,1,0.0,0,1,0,0,1,1,2
280,2026-03-11 02:00:05.155939+00:00,King St-Old Town,Mt Vern Sq,2026-03-10,22,1,0,0,YL,C13,...,0,0,0.5,0,0,1,1,1,0,2
281,2026-03-11 02:00:05.155939+00:00,King St-Old Town,Greenbelt,2026-03-10,22,1,0,0,YL,C13,...,0,0,0.5,0,0,1,1,1,0,2
282,2026-03-11 02:00:05.155939+00:00,King St-Old Town,Franconia-Springfield,2026-03-10,22,1,0,0,BL,C13,...,0,0,NaN,0,0,1,1,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333003,2026-03-18 18:30:16.166019+00:00,Rockville,Shady Grove,2026-03-18,14,2,0,0,RD,A14,...,0,0,NaN,0,1,0,0,1,1,2
333067,2026-03-18 18:30:16.166019+00:00,New Carrollton,Ashburn,2026-03-18,14,2,0,0,SV,D13,...,0,0,NaN,0,1,0,0,1,1,2
333129,2026-03-18 18:30:16.166019+00:00,Union Station,Shady Grv,2026-03-18,14,2,0,0,RD,B03,...,0,0,NaN,0,0,0,1,1,1,3
333130,2026-03-18 18:30:16.166019+00:00,Union Station,Shady Grv,2026-03-18,14,2,0,0,RD,B03,...,0,0,NaN,0,0,0,1,1,1,3


## 6. Add Previous and Next Stop (Locations) features

In [8]:
# Route sequences for each line retrieved from https://www.wmata.com/schedules/maps/upload/system-map-rail.pdf
RAIL_MAP = {
  "RD": ('shady', 'rockville', 'twinbrook', 'north bethesda', 'grosvenor', 'medical', 'bethesda', 'friendship', 'tenleytown', 'van ness', 'cleveland', 'woodley', 'dupont', 'farragut north', 'metro',
         'gallery', 'judiciary', 'union', 'noma', 'rhode', 'brookland', 'fort', 'takoma', 'silver', 'forest', 'wheaton', 'glenmont'),
  "BL": ('franconia', 'van dorn', 'king', 'braddock', 'potomac yard', 'reagan', 'crystal', 'pentagon city', 'pentagon', 'arlington', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'benning', 'capitol heights', 'addison', 'morgan boulevard', 'largo'),
  "GR": ('branch', 'suitland', 'naylor', 'southern', 'congress', 'anacostia', 'ballpark', 'waterfront', 'enfant', 'archives', 'gallery',
           'vernon', 'shaw', 'cardozo', 'columbia', 'georgia', 'fort', 'west hyattsville', 'hyattsville crossing', 'college', 'greenbelt'),
  "YL": (('huntington', 'eisenhower', 'king', 'braddock', 'potomac yard', 'reagan', 'crystal', 'pentagon city', 'pentagon', 'enfant', 'archives', 'gallery',
         'vernon', 'shaw', 'cardozo', 'columbia', 'georgia', 'fort', 'west hyattsville', 'hyattsville crossing', 'college', 'greenbelt'),
         ('huntington', 'eisenhower', 'king', 'braddock', 'potomac yard', 'reagan', 'crystal', 'pentagon city', 'pentagon', 'enfant', 'archives', 'gallery',
         'vernon')),
  "OR": ('vienna', 'dunn', 'west falls', 'east falls', 'ballston', 'virginia', 'clarendon', 'court', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'minnesota', 'deanwood', 'cheverly', 'landover', 'carrollton'),
  "SV": (('ashburn', 'loudoun', 'dulles', 'innovation', 'herndon', 'reston town', 'wiehle', 'spring hill', 'greensboro', 'tysons', 'mclean', 'east falls', 'ballston', 'virginia', 'clarendon', 'court', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'minnesota', 'deanwood', 'cheverly', 'landover', 'carrollton'),
         ('ashburn', 'loudoun', 'dulles', 'innovation', 'herndon', 'reston town', 'wiehle', 'spring hill', 'greensboro', 'tysons', 'mclean', 'east falls', 'ballston', 'virginia', 'clarendon', 'court', 'rosslyn', 'foggy', 'farragut west', 'mcpherson', 'metro',
         'federal triangle', 'smithsonian', 'enfant', 'federal center', 'capitol south', 'eastern', 'potomac ave', 'stadium', 'benning', 'capitol heights', 'addison', 'morgan boulevard', 'largo'))}

In [9]:
# Get the full name of a station from WMATA Stations dataset
@lru_cache(maxsize=None)
def resolve_station_fullname(name):
  station_names = tuple(df_stations['station_name'])
  return process.extractOne(name, station_names)[0]
    
# Map all station names of a line from RAIL_MAP dict to the station full name stated on WMATA Stations dataset
@lru_cache(maxsize=None)
def map_route_sequence(line_routes):
  mapped_route_sequence = []

  for route in line_routes:
    if isinstance(route, str): return tuple(map(resolve_station_fullname, line_routes))
    mapped_route_sequence.append(map_route_sequence(route))

  return tuple(mapped_route_sequence)

In [10]:
# Fix missing line codes on some stations served by lines that handle +1 routes (WMATA Stations Dataset)
def find_missing_stations(line):
  line_routes = map_route_sequence(RAIL_MAP[line])
  stations = df_stations[(df_stations == line).any(axis=1)]['station_name']

  missing_stations = []

  for route in line_routes:
    if isinstance(route, str): return set(route) - set(stations)
    missing_stations.extend(set(route) - set(stations))

  return missing_stations

def update_line_codes(line):
  lines_codes = ['line_code1', 'line_code2', 'line_code3', 'line_code4']

  for station_name in find_missing_stations(line):
    station = df_stations[(df_stations == station_name).any(axis=1)]

    for line_code in lines_codes:
      if station[line_code].isnull().values:
        df_stations.loc[station.index, line_code] = line
        break

update_line_codes('YL')
update_line_codes('SV')

In [11]:
# Flatten an array (or tuple)
def flatten(arr):
  flat_list = []
  for row in arr:
    if isinstance(row, str): return arr
    flat_list.extend(row)
      
  return flat_list

# Extract unique values from an array (or tuple)
def unique(arr):
  return list(dict.fromkeys(arr))

# Select the route that covers the location and destination stations on a line
@lru_cache(maxsize=None)
def select_route(line, location_name, destination_name):
  line_routes = RAIL_MAP[line]

  if line in ["YL", "SV"]: # Handle lines with +1 routes
    return [route for route in line_routes if (location_name in route) and (destination_name in route)][0]
  elif (location_name in  line_routes) and (destination_name in line_routes):
    return line_routes

# Get the shortname of a station based on the line route sequence established in RAIL_MAP dict
@lru_cache(maxsize=None)
def resolve_station_shortname(line, name):
  line_stations = tuple(unique(flatten(RAIL_MAP[line])))
  return process.extractOne(name, line_stations)[0]

# Find the previous or next stop from location based on destination station
@lru_cache(maxsize=None)
def get_stop_name(direction, line, location_name, destination_name): # direction: previous | next
  if "last" in [location_name.lower(), destination_name.lower()]:
    return '-'

  directions = { 'previous': 1, 'next': -1 }

  loc = resolve_station_shortname(line, location_name)
  dest = resolve_station_shortname(line, destination_name)

  route = select_route(line, loc, dest)

  loc_id = route.index(loc)
  dest_id = route.index(dest)

  if loc_id < dest_id:
    if (direction == 'previous') and (loc_id == 0):
      return route[loc_id + 1]
    else:
      return route[loc_id - directions[direction]]
  elif loc_id > dest_id:
    if (direction == 'previous') and (loc_id == (len(route) - 1)):
      return route[loc_id - 1]
    else:
      return route[loc_id + directions[direction]]
  else:
    return loc

# Get the code of a station from WMATA Stations dataset
@lru_cache(maxsize=None)
def get_station_code(line, station_name):
  # print(station_name)
  station_name = resolve_station_fullname(station_name)
  stations_identified_df = df_stations[df_stations['station_name'] == station_name]
  station_code = stations_identified_df[(stations_identified_df == line).any(axis=1)]['station_code']
  # print(station_code)
  if station_code.empty:
    return '-'
  else:
    return station_code.values[0]

In [12]:
# Add previous station code from location based on destination
df_features['previous_location_code'] = df_features.apply(lambda row: get_station_code(row['line'], get_stop_name('previous', row['line'],\
                                                                                                                  row['location_name'],\
                                                                                                                  row['destination_name'])), axis=1)

# Add next station code from location based on destination
df_features['next_location_code'] = df_features.apply(lambda row: get_station_code(row['line'], get_stop_name('next', row['line'],\
                                                                                                              row['location_name'],\
                                                                                                              row['destination_name'])), axis=1)

df_features

,collected_at,location_name,destination_name,date,hour,day_of_week,is_weekend,is_rush_hour,line,location_code,...,scheduled_headway_min,is_delayed,loc_has_parking,loc_is_transfer,loc_conn_vre,loc_conn_amtrak,loc_conn_mark,num_rails_conn,previous_location_code,next_location_code
0,2026-03-11 02:00:05.155939+00:00,Metro Center,Glenmont,2026-03-10,22,1,0,0,RD,A01,...,0.0,0,0,1,0,0,0,0,A02,B01
1,2026-03-11 02:00:05.155939+00:00,Court House,Vienna/Fairfax-GMU,2026-03-10,22,1,0,0,OR,K01,...,NaN,0,0,0,0,0,0,0,C05,K02
2,2026-03-11 02:00:05.155939+00:00,Benning Road,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G01,...,NaN,0,0,0,0,0,0,0,G02,D08
3,2026-03-11 02:00:05.155939+00:00,Addison Road-Seat Pleasant,Franconia-Springfield,2026-03-10,22,1,0,0,BL,G03,...,NaN,0,1,0,0,0,0,0,G04,G02
4,2026-03-11 02:00:05.155939+00:00,Van Dorn Street,Franconia-Springfield,2026-03-10,22,1,0,0,BL,J02,...,NaN,0,1,0,0,0,0,0,C13,J03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333137,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,0.5,0,0,1,0,0,0,0,F02,E01
333138,2026-03-18 18:30:16.166019+00:00,Gallery Pl-Chinatown,Greenbelt,2026-03-18,14,2,0,0,GR,F01,...,0.5,0,0,1,0,0,0,0,F02,E01
333139,2026-03-18 18:30:16.166019+00:00,Farragut North,Shady Grv,2026-03-18,14,2,0,0,RD,A02,...,NaN,0,0,0,0,0,0,0,A01,A03
333140,2026-03-18 18:30:16.166019+00:00,Takoma,Glenmont,2026-03-18,14,2,0,0,RD,B07,...,1.5,0,1,0,0,0,0,0,B06,B08


## 7. Save new features into existing features.csv

In [13]:
df_features.to_csv(PROJECT_DIR / "data" / 'features.csv', index=False)